# S2Gaussian Single-Scene Kaggle Pipeline (Run-All Ready)

This notebook is designed so you can set one scene name, click **Run All**, and train that scene end-to-end with visible progress output.

What this notebook guarantees:
- One scene per run (independent artifacts)
- Auto-check of required folders/files
- Dependency/bootstrap cell
- Live tqdm output in notebook during training
- Log file saved at the same time
- Optional split of compute: Gaussian training on GPU 0, Real-ESRGAN on GPU 1 (if available)

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil
import torch

# -------------------- Editable paths --------------------
# If you uploaded code as a Kaggle dataset, copy it under /kaggle/working first.
ROOT = Path('/kaggle/working')
REPO = ROOT / 'S2Gaussian'
REALESRGAN_REPO = ROOT / 'Real-ESRGAN'
DATA_ROOT = Path('/kaggle/input/hackathon-data')
OUTPUT_ROOT = ROOT / 'runs_s2'
LOG_ROOT = ROOT / 'logs'

# -------------------- Fixed scene list --------------------
SCENES = ['aeroplane', 'bike', 'buddha', 'cycle', 'face', 'firehydrant', 'still3', 'toy']

# -------------------- GPU routing --------------------
# Training always runs on GPU 0. Real-ESRGAN can use GPU 1 if present.
TRAIN_GPU_ID = 0
SR_GPU_ID = 1 if torch.cuda.device_count() > 1 else 0

# Make sure both GPUs are visible if present.
if torch.cuda.device_count() >= 2:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Optional memory-fragmentation guard.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:64,garbage_collection_threshold:0.8,expandable_segments:True')

LOG_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('CUDA device count:', torch.cuda.device_count())
print('Train GPU:', TRAIN_GPU_ID)
print('SR GPU:', SR_GPU_ID)
print('Repo path:', REPO)
print('Real-ESRGAN path:', REALESRGAN_REPO)
print('Data root:', DATA_ROOT)
print('Output root:', OUTPUT_ROOT)
print('Log root:', LOG_ROOT)

In [ ]:
def run_cmd(cmd, cwd=None, check=True):
    print(f'\n$ {cmd}')
    proc = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed ({proc.returncode}): {cmd}')

# 1) Validate code repos exist in expected locations.
assert REPO.exists(), f'Missing repo folder: {REPO}'
assert REALESRGAN_REPO.exists(), f'Missing Real-ESRGAN folder: {REALESRGAN_REPO}'

# 2) Validate input data exists.
assert DATA_ROOT.exists(), f'Missing dataset root: {DATA_ROOT}'
for s in SCENES:
    scene_dir = DATA_ROOT / s
    assert scene_dir.exists(), f'Missing scene folder: {scene_dir}'
    assert (scene_dir / 'images').exists(), f'Missing images folder: {scene_dir / "images"}'

# 3) Install Python dependencies.
run_cmd('python -m pip install -U pip setuptools wheel', check=True)
run_cmd('python -m pip install open3d timm plyfile opencv-python basicsr realesrgan', check=True)

# 4) Build/install CUDA extensions used by S2Gaussian.
# If already built, these commands are typically fast/no-op.
run_cmd('python -m pip install --no-build-isolation -e S2Gaussian/fsgs/submodules/diff-gaussian-rasterization-confidence', cwd=ROOT, check=True)
run_cmd('python -m pip install --no-build-isolation -e S2Gaussian/fsgs/submodules/simple-knn', cwd=ROOT, check=True)

# 5) Validate critical SR weight file.
sr_weight = REALESRGAN_REPO / 'RealESRGAN_x4plus.pth'
assert sr_weight.exists(), f'Missing SR weight file: {sr_weight}'
print('\nBootstrap complete.')

In [ ]:
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda devices:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

# Validate train script and key folders.
assert (REPO / 'scripts' / 'train_s2gs.py').exists(), 'train_s2gs.py missing'
assert (REALESRGAN_REPO / 'RealESRGAN_x4plus.pth').exists(), 'RealESRGAN_x4plus.pth missing'

# Validate that sr_gpu_id choice is legal.
assert SR_GPU_ID < max(torch.cuda.device_count(), 1), 'SR GPU id is out of range'
print('Preflight check passed.')

## Preflight Check (Must Pass)

This verifies CUDA, imports, and critical file paths before training starts.

## Bootstrap (One-Time Per Kaggle Session)

Run this cell once after startup. It installs dependencies, validates required folders/files, and attempts CUDA extension builds.

## Load Scene Data

Set the active scene name below and point the scene-specific input paths at Kaggle input folders. Each run should process exactly one scene and write its own outputs independently.

In [ ]:
ACTIVE_SCENE = 'bike'  # Change once, then click Run All.
SOURCE_PATH = DATA_ROOT / ACTIVE_SCENE
MODEL_PATH = OUTPUT_ROOT / ACTIVE_SCENE
LOG_FILE = LOG_ROOT / f'{ACTIVE_SCENE}_train.log'

print('Active scene:', ACTIVE_SCENE)
print('Source path:', SOURCE_PATH)
print('Model path:', MODEL_PATH)
print('Log file:', LOG_FILE)

assert ACTIVE_SCENE in SCENES, f'Unknown scene: {ACTIVE_SCENE}'
assert SOURCE_PATH.exists(), f'Missing scene folder: {SOURCE_PATH}'
assert (SOURCE_PATH / 'images').exists(), f'Missing images folder: {SOURCE_PATH / "images"}'
assert (SOURCE_PATH / 'sparse' / '0').exists(), f'Missing COLMAP sparse folder: {SOURCE_PATH / "sparse" / "0"}'
MODEL_PATH.mkdir(parents=True, exist_ok=True)

## Initialize Gaussian Splatting Model

This section keeps the run scene-local. It uses one scene directory, one model directory, and one log file per execution.

In [ ]:
# Safe defaults: start with moderate iterations, then scale up after one successful scene.
stage1_iterations = 6000
stage2_iterations = 500
sr_model_path = REALESRGAN_REPO / 'RealESRGAN_x4plus.pth'

train_script = REPO / 'scripts' / 'train_s2gs.py'

command = [
    'python', str(train_script),
    '--source_path', str(SOURCE_PATH),
    '--images', 'images',
    '--model_path', str(MODEL_PATH),
    '--stage1_iterations', str(stage1_iterations),
    '--stage2_iterations', str(stage2_iterations),
    '--eval',
    '--disable_densification',
    '--sr_model_path', str(sr_model_path),
    '--sr_tile', '128',
    '--sr_tile_pad', '20',
    '--sr_gpu_id', str(SR_GPU_ID),
]

print('Training command:')
print(' '.join(command))

## Train Model on Single Scene

Run this cell once per scene. Do not loop all scenes in the notebook if you want clean, independent artifacts.

In [ ]:
# Stream output to notebook (tqdm visible) and log file at the same time.
with LOG_FILE.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)

    process.wait()

print('Return code:', process.returncode)
assert process.returncode == 0, f'Training failed for {ACTIVE_SCENE}. Check {LOG_FILE}'

## Save Model Weights and Outputs

Each scene keeps its own checkpoint and outputs under /kaggle/working/runs_s2/<scene> so nothing is overwritten across scenes.

In [ ]:
scene_output_dir = OUTPUT_ROOT / ACTIVE_SCENE
scene_output_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    'log_file': str(LOG_FILE),
    'model_path': str(MODEL_PATH),
    'scene_output_dir': str(scene_output_dir),
}
with (scene_output_dir / 'metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(artifacts, handle, indent=2)

print('Saved scene artifacts to:', scene_output_dir)

## Validate and Evaluate Scene Results

After training, render the scene outputs and compute metrics for the selected scene. Save the evaluation report alongside the scene artifacts.

In [ ]:
# Replace this with the scene-specific render/eval command that your repo expects.
# The goal is to keep each scene independent, then combine the rendered outputs later.
render_command = [
    'python', str(REPO / 'fsgs' / 'render.py'),
    '--source_path', str(SOURCE_PATH),
    '--model_path', str(MODEL_PATH),
    '--images', 'images',
    '--quiet',
]

print('Render command:')
print(' '.join(render_command))

# Placeholder for evaluation metrics aggregation.
evaluation_report = scene_output_dir / 'evaluation.json'
with evaluation_report.open('w', encoding='utf-8') as handle:
    json.dump({'scene': ACTIVE_SCENE, 'status': 'training_complete', 'note': 'Add scene render/eval here'}, handle, indent=2)

print('Saved evaluation report to:', evaluation_report)

## Combine Outputs Later

After all 8 scenes finish, copy or download each scene folder, render the test views, and merge the scene outputs into one submission file on your local machine or in a final Kaggle notebook.

In [ ]:
# Example combine step (local or final Kaggle notebook):
# 1. Gather per-scene render folders.
# 2. Map scene outputs to the test filenames.
# 3. Run imgs2csv.py to produce submission.csv.

combine_plan = {
    'scenes': SCENES,
    'per_scene_root': str(OUTPUT_ROOT),
    'submission_note': 'Use scene folders independently, then merge at the end.',
}
print(json.dumps(combine_plan, indent=2))

## End-to-End Setup and Run Instructions

Follow exactly in this order:

1. In Kaggle Notebook settings, select GPU acceleration. If two GPUs are available, this notebook will use GPU 0 for training and GPU 1 for Real-ESRGAN.
2. Ensure these folders exist before running cells:
   - `/kaggle/working/S2Gaussian`
   - `/kaggle/working/Real-ESRGAN`
   - `/kaggle/input/hackathon-data/<scene>` for each scene.
3. Confirm `RealESRGAN_x4plus.pth` exists at `/kaggle/working/Real-ESRGAN/RealESRGAN_x4plus.pth`.
4. Set `ACTIVE_SCENE` in the scene selection cell.
5. Click **Run All**.
6. Watch live tqdm output in the training cell. Logs are also saved to `/kaggle/working/logs/<scene>_train.log`.
7. Trained outputs are stored per scene in `/kaggle/working/runs_s2/<scene>`.
8. Download that scene folder and log after completion.
9. Repeat for the other scenes by changing only `ACTIVE_SCENE` and rerunning.
10. After all 8 scenes are done, combine render outputs and generate final `submission.csv`.

Troubleshooting:
- If Stage 2 OOM occurs, reduce `--sr_tile` from `128` to `96` or `64` in the command cell.
- If speed is too slow, increase `--sr_tile` gradually (for example `160` or `192`) only if memory remains stable.
- Keep `--disable_densification` enabled for stability.
- Do not enable `--sr_fp32` unless absolutely necessary, since it increases memory usage.